In [0]:
import yaml

# Loading the config file

CONFIG_PATH = "/Workspace/Users/avranilset@gmail.com/capstone-project/config/config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Getting the Catalog and Gold Schema Path

CATALOG = config["databricks"]["catalog"]
GOLD_DB = config["databricks"]["schemas"]["gold"]

# Getting the table paths

DIM_CUSTOMER = config["gold"]["dim_customer"]
DIM_PRODUCT = config["gold"]["dim_product"]
FACT_SALES = config["gold"]["fact_sales"]

# Setting up snowflake credentials

SNOWFLAKE_OPTIONS = {
    "sfUrl": dbutils.secrets.get("snowflake", "account") + ".snowflakecomputing.com",
    "sfUser": dbutils.secrets.get("snowflake", "user"),
    "sfPassword": dbutils.secrets.get("snowflake", "password"),
    "sfDatabase": config["snowflake"]["database"],
    "sfSchema": config["snowflake"]["schema"],
    "sfWarehouse": config["snowflake"]["warehouse"],
    "sfRole": config["snowflake"]["role"],
}

print("SNOWFLAKE LOADING")


# Function for loading to snowflake

def load_to_snowflake(source_table: str, target_table: str) -> None:
    df = spark.table(source_table)
    count = df.count()
    (
        df.write.format("snowflake")
        .options(**SNOWFLAKE_OPTIONS)
        .option("dbtable", target_table)
        .mode("overwrite")
        .save()
    )
    print(f"{target_table}: {count:,} moved to Snowflake")




# Calling the function for each table

load_to_snowflake(f"{CATALOG}.{GOLD_DB}.dim_customer", "DIM_CUSTOMER")
load_to_snowflake(f"{CATALOG}.{GOLD_DB}.dim_product", "DIM_PRODUCT")
load_to_snowflake(f"{CATALOG}.{GOLD_DB}.fact_sales", "FACT_SALES")

print("\nSNOWFLAKE LOADING — COMPLETE")